# 00 - Setup: Carga de Dados no SQL Server

Este notebook carrega os arquivos CSV da pasta `data/` para o banco de dados **EcommerceDB** no SQL Server 2025.

**Fonte dos dados:** [Brazilian Parliamentary Expenses Datasets (BPED)](https://www.kaggle.com/datasets/joaopauloschiavon/brazilian-parliamentary-expenses-datasets-bped) — Kaggle.  
Dados baseados na **CEAP** (Cota para o Exercício da Atividade Parlamentar) da Câmara dos Deputados do Brasil, com amostra de 20–30 registros por tabela para fins didáticos.

**Tabelas:**
- `partidos` — partidos políticos
- `parlamentares` — deputados federais
- `categorias_despesa` — tipos de despesa (alimentação, combustível, etc.)
- `fornecedores` — empresas e pessoas que prestaram serviços
- `despesas` — registros de reembolso do CEAP

**Pré-requisitos:**
- Docker Compose rodando (`docker compose up -d`)
- SQL Server acessível na porta `1433`
- ODBC Driver 18 instalado

## 1. Configuração e Conexão

In [ ]:
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv

load_dotenv(override=True)

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

print(f'Servidor: {DB_SERVER}:{DB_PORT}')
print(f'Database: {DB_DATABASE}')

In [ ]:
conn_master = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor_master = conn_master.cursor()
print('Conectado ao SQL Server (master) com sucesso!')

## 2. Criar Database

In [ ]:
cursor_master.execute(f"""
    IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{DB_DATABASE}')
    BEGIN
        CREATE DATABASE [{DB_DATABASE}]
    END
""")
print(f'Database [{DB_DATABASE}] criado/verificado com sucesso!')
cursor_master.close()
conn_master.close()

In [ ]:
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor = conn.cursor()
print(f'Conectado ao [{DB_DATABASE}] com sucesso!')

## 3. Criar Tabelas

In [ ]:
ddl_statements = [
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'partidos')
    CREATE TABLE partidos (
        cd_partido   INT PRIMARY KEY,
        sg_partido   VARCHAR(20)  NOT NULL,
        nm_partido   VARCHAR(100) NOT NULL
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'parlamentares')
    CREATE TABLE parlamentares (
        cd_parlamentar  INT PRIMARY KEY,
        nm_parlamentar  VARCHAR(200) NOT NULL,
        sg_uf           CHAR(2)      NOT NULL,
        cd_partido      INT          NOT NULL,
        nu_legislatura  INT          NOT NULL
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'categorias_despesa')
    CREATE TABLE categorias_despesa (
        cd_categoria      INT PRIMARY KEY,
        ds_categoria      VARCHAR(200) NOT NULL,
        ds_especificacao  VARCHAR(200)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'fornecedores')
    CREATE TABLE fornecedores (
        cd_fornecedor  INT PRIMARY KEY,
        nm_fornecedor  VARCHAR(200) NOT NULL,
        nr_cnpj_cpf    VARCHAR(20),
        ds_tipo        VARCHAR(50)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'despesas')
    CREATE TABLE despesas (
        cd_despesa         INT PRIMARY KEY,
        cd_parlamentar     INT          NOT NULL,
        cd_categoria       INT          NOT NULL,
        cd_fornecedor      INT          NOT NULL,
        nu_mes             INT          NOT NULL,
        nu_ano             INT          NOT NULL,
        dt_emissao         DATE,
        vl_documento       DECIMAL(12,2) NOT NULL,
        vl_glosa           DECIMAL(12,2),
        vl_liquido         DECIMAL(12,2) NOT NULL,
        nr_documento       VARCHAR(50),
        ind_tipo_documento INT
    )
    """
]

for ddl in ddl_statements:
    cursor.execute(ddl)

print('Todas as tabelas foram criadas com sucesso!')

## 4. Carregar Dados dos CSVs

In [ ]:
tabelas = ['partidos', 'parlamentares', 'categorias_despesa', 'fornecedores', 'despesas']

data_dir = os.path.join(os.getcwd(), '..', 'data')

for tabela in tabelas:
    csv_path = os.path.join(data_dir, f'{tabela}.csv')

    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]

    if count > 0:
        print(f'  ⏭️  {tabela}: já contém {count} registros, pulando...')
        continue

    df = pd.read_csv(csv_path)

    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].str.strip()

    cols = ', '.join(df.columns)
    placeholders = ', '.join(['?' for _ in df.columns])
    insert_sql = f'INSERT INTO {tabela} ({cols}) VALUES ({placeholders})'

    data = [tuple(None if pd.isna(v) else v for v in row) for row in df.itertuples(index=False)]

    cursor.executemany(insert_sql, data)
    print(f'  ✅ {tabela}: {len(data)} registros inseridos')

print('\n🎉 Carga de dados concluída!')

## 5. Validação

In [ ]:
print(f'{"Tabela":<25} {"Registros":>10}')
print('-' * 37)

for tabela in tabelas:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    print(f'{tabela:<25} {count:>10}')

print('\n✅ Validação concluída!')

In [ ]:
for tabela in ['partidos', 'parlamentares', 'despesas']:
    print(f'\n--- {tabela.upper()} (primeiros 5 registros) ---')
    df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
    print(df_sample.to_string(index=False))

In [ ]:
cursor.close()
conn.close()
print('Conexão encerrada.')